In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from langchain_core.runnables import RunnableParallel,RunnableBranch,RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [6]:
model = ChatOllama(
    model= "gpt-oss:120b-cloud"
)

parser = StrOutputParser()

class feedback(BaseModel):
    sentiment:Literal['positive','negative']=Field(description="The sentiment of the feedback, must be either positive or negative")

parser2 = PydanticOutputParser(
    pydantic_object=feedback
)

prompt1 = PromptTemplate(
    template="classify the sentiment of the following text into positive or negative {feedback}{format_instructions}",
    input_variables= ['feedback'],
    partial_variables={"format_instructions":parser2.get_format_instructions()}
)

classifier_chain = prompt1|model|parser2

prompt2 = PromptTemplate(
    template="Write an appropriate response to this positive feedback{feedback}",
    input_variables= ['feedback']   
)

prompt3 = PromptTemplate(
    template="Write an appropriate response to this negative feedback{feedback}",
    input_variables= ['feedback']   
)


branch_chain = RunnableBranch(
    (lambda x:x.sentiment=="positive",prompt2|model|parser),
    (lambda x:x.sentiment=="negative",prompt3|model|parser),
    RunnableLambda(lambda x:"No valid sentiment found")
)

chain = classifier_chain|branch_chain

response = chain.invoke({"feedback":"This is a beautiful place which has a lot of amazing historical insights and people are also very good. Thankyou for having us here"})
print(response)

Thank you so much for your kind words! We're thrilled to hear that you had a positive experience, and your feedback really brightens our day. Knowing that our effort made a difference motivates us to keep improving and delivering the best possible service. If there’s anything else we can do for you or any suggestions you have, please don’t hesitate to let us know. Thanks again for your support! 🙏✨
